In [2]:
%reload_ext autoreload
%autoreload 2

import time
from datetime import datetime
import copy
from pathlib import Path
from tqdm import tqdm
from torch_geometric.loader import DataLoader
import os.path as osp
import torch
import torch.nn as nn
import torch
from torch_geometric.loader import DataLoader
from src.dataset import Crystals
from src.loss import MSELoss, L1Loss, CosSimLoss, KLDivLoss, CombLoss
from src.models import MDNet

/workspace/pytorch_GNN/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


In [3]:
device = torch.device('cuda')
# loss_fn = MSELoss()
loss_fn = MSELoss()
loss_name = repr(loss_fn)
now = datetime.now()
dt_string = now.strftime("%Y%m%d-%H%M%S")
dataset_path = '../data/processed/v6.pt'
dataset = Crystals(dataset_path)

train_dataset, val_dataset, test_dataset = dataset.split()
train_loader = DataLoader(train_dataset, batch_size=64)
val_loader = DataLoader(val_dataset, batch_size=64)

fn = KLDivLoss()
def val(model, val_dataloader):
    model.eval()
    with torch.inference_mode():
        mse = 0
        for data in val_dataloader:
            data = data.to(device)
            pred = model(data)

            mse += fn(pred, data.y)
        return mse / len(val_dataloader)

train_loss = []
val_loss = []
display_epochs = 10
best_val_loss = 0.007
model_save_path = None

In [14]:
from src.models import Mace
args = {
    "hidden_size": 256,
    "dropout": 0
}
model = Mace(args).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 1e-5)
for batch in train_loader:
    batch = batch.to(device)
    
    if torch.isnan(batch.pos).any() or torch.isnan(batch.wl).any():
        print("batch")
        break
        
    out = model(batch)
    loss = criterion(out, batch.y)
    
    if torch.isnan(loss):
        print("loss")
        break
        
    optimizer.zero_grad()
    loss.backward()
    
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    if torch.isnan(grad_norm):
        print("grad")
        break
    optimizer.step()

/workspace/pytorch_GNN/.venv/lib/python3.12/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.warn(
/workspace/pytorch_GNN/.venv/lib/python3.12/site-packages/mace/modules/blocks.py:316: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(atomic_energies, dtype=torch.get_default_dtype()),
/workspace/pytorch_GNN/.venv/lib/python3.12/site-packages/torch/jit/_check.py:178: UserWarning: The TorchScript type system doesn't support instance-level annotations on empty non-base types in `__init__`. Instead, either 1) use a type annotation in the class body, or 2) wrap the type in `torch.jit.Attribute`.
  warnings.

RuntimeError: Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed). Saved intermediate values of the graph are freed when you call .backward() or autograd.grad(). Specify retain_graph=True if you need to backward through the graph a second time or if you need to access saved tensors after calling backward.

In [19]:
import torch
torch.cuda.empty_cache()

In [18]:
a = next(iter(train_loader))
a

DataBatch(x=[665, 8], edge_index=[2, 9584], pos=[665, 3], element=[64], z=[665], dist=[9584], y=[64, 266], mineral=[64], wl=[64], batch=[665], ptr=[65])